# The hashes behind the boundaries: FNV-1a vs the mask rule vs buzhash

The other notebooks use three names that are easy to blur together, and one of them isn't even a
hash. This notebook takes them apart:

- **FNV-1a** — a *hash*: an XOR-then-multiply **accumulator** over bytes.
- **the mod / mask test** — *not a hash*: a **decision rule** (`(h & patt) == target`) that turns
  any uniform hash value into a boundary coin with probability `2^-k`.
- **buzhash** — a *hash*: a table + rotate + XOR **rolling window**, built so the oldest byte can
  be removed exactly.
- (**gear**, the FastCDC-lineage candidate from the engine's
  [boundary study](../docs/write-ups/chunker-boundary-detection-study.md), joins for one section
  because it teaches the sharpest lesson about *which bits you mask*.)

One quantity organizes all of them: the **influence horizon** — how many bytes back an input can
sit and still change the hash state. It is the hidden variable behind every pro and con here,
and every claim below is measured.

Also worth separating before starting: a prolly store uses hashing for **two different jobs**.
*Addresses* (chunk ids, commit ids) need collision resistance — that is the engine's
[hash-function study](../docs/write-ups/hash-function-study.md) (SHA-2 vs BLAKE, hardware
instructions, 20-byte truncation). *Boundaries* need something much weaker: fast, deterministic,
uniform-enough low bits. None of the functions here is collision-resistant, and none needs to be.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import prolly_chunking as pc

plt.rcParams.update({"figure.figsize": (7, 3.0), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})
rng = np.random.default_rng(0)
DATA = pc.sha256_counter_stream(200_000)
print("ready;", len(DATA), "corpus bytes")

ready; 200000 corpus bytes


## 1. FNV-1a: an accumulator

```
h = seed
for b in bytes:  h = ((h ^ b) * PRIME) mod 2^32
```

Every byte is folded in and then *smeared upward* by the multiply (an odd prime: a bijection mod
2^32, and low-end-local — the sealed-lane property from the first notebook). Two measurements
tell you what it is and isn't:

**Is the boundary coin fair?** The mask rule only needs the low bits to be uniform over real
inputs. Frequency of each low-3-bit value over 100k sequential keys:

In [2]:
lows = np.array([pc.key_hash(k) & 7 for k in range(100_000)])
counts = np.bincount(lows, minlength=8)
print("low-3-bit value counts over 100k sequential keys (expect ~12500 each):")
print(" ", counts.tolist())
dev = np.abs(counts - 12_500).max() / 12_500
print(f"worst deviation from uniform: {dev:.1%}")
assert dev < 0.05
print("→ the coin is fair on exactly the boring input (sequential ints) that would")
print("  expose a weak hash — this is what 'good enough for boundaries' means")

low-3-bit value counts over 100k sequential keys (expect ~12500 each):
  [12500, 12500, 12500, 12500, 12500, 12500, 12500, 12500]
worst deviation from uniform: 0.0%
→ the coin is fair on exactly the boring input (sequential ints) that would
  expose a weak hash — this is what 'good enough for boundaries' means


**How well does it mix?** Flip one input bit, count output bits that flip. A cryptographic hash
targets 16 of 32 from every input position ('avalanche'); FNV-1a makes no such promise — later
bytes pass through fewer multiplies, so they are smeared less. Measure the mean flip count per
input byte position (4-byte keys):

In [3]:
def avalanche_by_byte(trials=2000):
    out = np.zeros(4)
    for t in range(trials):
        k = int(rng.integers(0, 2**32))
        base = pc.fnv1a(k.to_bytes(4, "little"))
        for pos in range(4):
            flipped = k ^ (1 << (8 * pos + int(rng.integers(0, 8))))
            out[pos] += bin(base ^ pc.fnv1a(flipped.to_bytes(4, "little"))).count("1")
    return out / trials

av = avalanche_by_byte()
for pos, a in enumerate(av):
    print(f"flip a bit in byte {pos} ({'first hashed' if pos == 0 else 'last hashed' if pos == 3 else '…'}):"
          f"  {a:5.2f} of 32 output bits flip")
print("→ imperfect and position-dependent — and irrelevant for the boundary coin, which")
print("  only needs the low bits fair. (For ADDRESSES this would be disqualifying;")
print("  addresses use SHA-2.)")

flip a bit in byte 0 (first hashed):  14.63 of 32 output bits flip
flip a bit in byte 1 (…):  15.14 of 32 output bits flip
flip a bit in byte 2 (…):  12.42 of 32 output bits flip
flip a bit in byte 3 (last hashed):   8.95 of 32 output bits flip
→ imperfect and position-dependent — and irrelevant for the boundary coin, which
  only needs the low bits fair. (For ADDRESSES this would be disqualifying;
  addresses use SHA-2.)


## 2. The mask is the coin, the hash is only the dice-roller

`(h & (2^k − 1)) == 0` fires with probability `2^-k` *whatever* uniform hash produced `h` — the
chunk-size distribution comes from the **rule**, not the hash. Same rule, three different
sources, indistinguishable geometry:

In [4]:
k = 6

def sizes_from_hits(hits):
    idx = np.flatnonzero(hits)
    return np.diff(np.concatenate(([-1], idx)))

fnv_hits = np.array([(pc.key_hash(i) & ((1 << k) - 1)) == 0 for i in range(60_000)])

bz = pc.BuzHash(pc.PROD_WINDOW)
buz_hits = np.array([(bz.hash_byte(b) & ((1 << k) - 1)) == 0 for b in DATA[:60_000]])

rand_hits = rng.integers(0, 1 << k, size=60_000) == 0

print(f"{'source':>14} {'mean size':>10} {'std':>8}   (rule: low {k} bits == 0 → target {1 << k})")
for name, hits in [("FNV per key", fnv_hits), ("buzhash", buz_hits), ("pure random", rand_hits)]:
    s = sizes_from_hits(hits)
    print(f"{name:>14} {s.mean():>10.1f} {s.std():>8.1f}")
print("→ identical geometric statistics: the hash only has to be a fair, DETERMINISTIC")
print("  dice-roller; determinism is what 'pure random' lacks and content-defined needs")

        source  mean size      std   (rule: low 6 bits == 0 → target 64)
   FNV per key       64.0     15.9
       buzhash       64.9     63.2
   pure random       63.0     60.3
→ identical geometric statistics: the hash only has to be a fair, DETERMINISTIC
  dice-roller; determinism is what 'pure random' lacks and content-defined needs


### 2b. Could SHA-256 decide boundaries? Yes — and that's the point

A natural objection: if the mask rule works on any fair hash, could you chunk with a
*cryptographic* hash? Completely — run SHA-256 over the last W bytes at every position and mask
its low bits. The geometry is identical and, because the input is a bounded window, the horizon
is W, so it even heals edits like buzhash does. What it costs is **re-hashing W bytes at every
position** — O(W) work per byte where a rolling hash pays O(1). Measure both:

In [5]:
import hashlib, time

def windowed_sha_hits(data, W, k=6):
    mask = (1 << k) - 1
    return np.array([
        (int.from_bytes(hashlib.sha256(data[i - W:i]).digest()[:4], "little") & mask) == 0
        for i in range(W, len(data))])

def buz_hits_W(data, W, k=6):
    mask = (1 << k) - 1
    bz = pc.BuzHash(W)
    return np.array([(bz.hash_byte(b) & mask) == 0 for b in data])

SAMPLE = DATA[:40_000]

# Geometry first: same mean chunk size from the crypto hash as from the roller.
s_sha = sizes_from_hits(windowed_sha_hits(SAMPLE, W=67))
s_buz = sizes_from_hits(buz_hits_W(SAMPLE, W=67))
print(f"windowed SHA-256 mean chunk {s_sha.mean():5.1f} · rolling buzhash {s_buz.mean():5.1f}"
      f"  (target 64) — same geometry\n")

# Cost: a cross-language constant factor would be a LIE here (hashlib is C,
# our buzhash is pure Python — the harness would measure the languages, not
# the algorithms). The honest instrument is SCALING IN W: windowed work grows
# with the window, rolling work does not.
print(f"{'W':>6} {'windowed SHA  ns/byte':>22} {'rolling buzhash  ns/byte':>25}")
for W in (67, 256, 1024, 4096):
    t0 = time.perf_counter(); windowed_sha_hits(SAMPLE, W); t_sha = time.perf_counter() - t0
    t0 = time.perf_counter(); buz_hits_W(SAMPLE, W);        t_buz = time.perf_counter() - t0
    print(f"{W:>6} {t_sha / len(SAMPLE) * 1e9:>22.0f} {t_buz / len(SAMPLE) * 1e9:>25.0f}")
print("\n→ windowed cost grows ~linearly with W; rolling cost is flat — O(W) vs O(1)")
print("  per byte. (At W=67 with a C-vs-Python harness the constants even favor SHA;")
print("  the scaling, not the constant, is why rolling hashes exist.)")

windowed SHA-256 mean chunk  66.6 · rolling buzhash  62.0  (target 64) — same geometry

     W  windowed SHA  ns/byte  rolling buzhash  ns/byte


    67                    579                       646


   256                    641                       603


  1024                    993                       644


  4096                   2291                       643

→ windowed cost grows ~linearly with W; rolling cost is flat — O(W) vs O(1)
  per byte. (At W=67 with a C-vs-Python harness the constants even favor SHA;
  the scaling, not the constant, is why rolling hashes exist.)


So the division of labor is sharper than "fast hashes for boundaries":

1. **Any fair, deterministic hash can make the boundary decision** — the rule doesn't care.
2. **The horizon is the actual design requirement**: whole-item (per-element chunking) or a
   bounded window (byte streams, for edit healing). A cryptographic hash satisfies it too, if
   you feed it a window.
3. **Speed only picks the cheapest hash that delivers the chosen horizon**: FNV per element
   (no window needed, so nothing to roll), a rolling hash per byte stream (it makes the window
   O(1) per byte instead of O(W)).

And with the engine's numbers attached, even point 3 is smaller than it looks: the whole
boundary function is ~1.6% of ingest (the boundary study's gate verdict), while the *address*
hash — which genuinely must be cryptographic — is ~10× that share (the hash-function study
measured −9.3% of whole ingest just from SHA-512 → hardware SHA-256). Where speed actually
bought something measurable was the address hash, not the boundary hash.

## 3. The influence horizon — the one number that separates them

Definition: flip a byte at distance `d` before the point where the hash state is read; does the
state change? The largest `d` that can still matter is the hash's **horizon**.

- **FNV-1a**: every round is a bijection (XOR, then multiply by an odd prime), so *any* earlier
  flip changes the final state. Horizon **∞** — an accumulator never forgets. That is exactly
  why it cannot roll, and why the per-element scheme hashes each item *whole* (the item is the
  horizon).
- **buzhash**: a byte's contribution is `rot^age(T[b])`; the window buffer removes it exactly at
  age W. Horizon **= W, by construction** — bounded *and tunable*, which is what edit healing
  (first notebook, §6) actually tunes.
- **gear** (`h = (h << 1) + T[b] mod 2^32`): flip a byte and the state *difference* evolves as
  `delta → delta << 1 mod 2^32` — after 32 shifts the difference falls off the top **exactly**.
  Horizon **≤ 32, structurally** — the state width is the window, no buffer needed. And bit `j`
  of the state can only see the last `j+1` bytes, so *masking low bits gives a horizon of the
  mask width*: a low-13-bit gear rule is decided by 13 bytes. FastCDC masks gear's **high** bits
  for precisely this reason.

Measure all of it — change *rate* at the state-read point as a function of flip distance:

In [6]:
def change_rate(state_fn, d, trials=60, length=200):
    changed = 0
    for t in range(trials):
        buf = bytearray(rng.integers(0, 256, size=length, dtype=np.uint8).tobytes())
        base = state_fn(bytes(buf))
        pos = length - 1 - d
        buf[pos] ^= 1 << int(rng.integers(0, 8))
        changed += state_fn(bytes(buf)) != base
    return changed / trials

def fnv_state(data):
    return pc.fnv1a(data)

def buz_state(data):
    bz = pc.BuzHash(pc.PROD_WINDOW)
    for b in data:
        bz.hash_byte(b)
    return bz.sum32()

GEAR = [int(x) for x in np.random.default_rng(99).integers(0, 2**32, size=256, dtype=np.uint64)]

def gear_state(data):
    h = 0
    for b in data:
        h = ((h << 1) + GEAR[b]) & 0xFFFFFFFF
    return h

def gear_low13(data):  return gear_state(data) & 0x1FFF
def gear_high13(data): return gear_state(data) >> 19

print(f"{'d':>4} {'FNV':>6} {'buzhash':>8} {'gear':>6} {'gear&low13':>11} {'gear>>19':>9}")
for d in (5, 12, 13, 31, 32, 40, 66, 67, 80, 120):
    row = [change_rate(f, d) for f in (fnv_state, buz_state, gear_state, gear_low13, gear_high13)]
    print(f"{d:>4} " + " ".join(f"{r:>{w}.0%}" for r, w in zip(row, (6, 8, 6, 11, 9))))
print("\nhorizons: FNV = ∞ (never forgets) · buzhash = 67 (the window, exactly)")
print("          gear = 32 (state width)   · gear low-13 mask = 13 (mask width!)")

   d    FNV  buzhash   gear  gear&low13  gear>>19
   5   100%     100%   100%         98%      100%
  12   100%     100%   100%         63%      100%
  13   100%     100%   100%          0%      100%
  31   100%     100%    52%          0%       57%
  32   100%     100%     0%          0%        0%
  40   100%     100%     0%          0%        0%
  66   100%     100%     0%          0%        0%
  67   100%       0%     0%          0%        0%
  80   100%       0%     0%          0%        0%
 120   100%       0%     0%          0%        0%

horizons: FNV = ∞ (never forgets) · buzhash = 67 (the window, exactly)
          gear = 32 (state width)   · gear low-13 mask = 13 (mask width!)


## 4. Why XOR can roll and multiply cannot

Buzhash's state is a plain XOR of independent per-byte terms, `⊕ rot^age(T[b])` — XOR is
invertible and each term depends only on its *own* byte and age, so the splitter can XOR the
oldest term back out. FNV's multiply entangles every byte with everything that came after it;
there is no term to remove.

The same rotation structure buys the removal — and sells a **structured collision**: rotation
age works mod 32, so two bytes exactly 32 positions apart contribute `rot^a(T[b₁]) ⊕
rot^a(T[b₂])`. Swap them and the hash cannot tell:

In [7]:
w = bytearray(rng.integers(0, 256, size=pc.PROD_WINDOW, dtype=np.uint8).tobytes())
i = 10
w2 = bytearray(w); w2[i], w2[i + 32] = w2[i + 32], w2[i]      # swap bytes 32 apart
assert bytes(w) != bytes(w2)

print(f"buzhash(original) = {buz_state(bytes(w)):#010x}")
print(f"buzhash(swapped)  = {buz_state(bytes(w2)):#010x}   ← identical: rotation ages alias mod 32")
assert buz_state(bytes(w)) == buz_state(bytes(w2))
print(f"fnv1a distinguishes them: {fnv_state(bytes(w)):#010x} vs {fnv_state(bytes(w2)):#010x}")
assert fnv_state(bytes(w)) != fnv_state(bytes(w2))
print("\n→ a structured collision class, accepted as noise: it shifts the odd boundary,")
print("  never breaks determinism — and boundary hashes never promised collision resistance")

buzhash(original) = 0x2ed897e9
buzhash(swapped)  = 0x2ed897e9   ← identical: rotation ages alias mod 32
fnv1a distinguishes them: 0x540cda5e vs 0x97f7496a

→ a structured collision class, accepted as noise: it shifts the odd boundary,
  never breaks determinism — and boundary hashes never promised collision resistance


## 5. Pros and cons, earned

| | **FNV-1a** (accumulator) | **buzhash** (rolling window) | **gear** (shift accumulator) |
|---|---|---|---|
| horizon | ∞ — never forgets | **= W exactly**, tunable (buffer) | ≤ 32 (state width); = mask width on low bits |
| can roll? | no — multiply entangles | **yes** — XOR terms removable | no removal needed: shift *is* the forgetting |
| cost / byte | 1 mul + 1 xor | 2 rot + 2 xor + table load + buffer write | 1 shift + 1 add + table load |
| coin fairness | low bits uniform (measured §1) | uniform (measured §2) | low bits = short horizon → mask **high** bits |
| collision structure | none exploitable found here; weak tail mixing (§1) | swap-at-32 aliasing (§4) | ages alias by shift; carries only travel up |
| right regime | **per-element**: the item is the horizon | **byte streams**: horizon must outlive any item | byte streams where speed wins and W≤32 is enough |
| in this repo | the pedagogy scheme's `key_hash` | **production** (`RollingHashSplitter`, W=67) | study candidate B — see the verdict below |

Three take-aways:

1. **Pick the horizon first, then the hash.** Per-element chunking needs no window — FNV (or any
   fast hash) is right. Byte-stream chunking *is* the choice of a horizon; buzhash buys an exact,
   tunable one for the price of a buffer, gear a fixed ≤32 one for almost nothing.
2. **The mask rule is common property.** Geometry (target size, staircase, clamps) lives in the
   rule and composes with any of these hashes — which is why the engine could benchmark
   direct-mask vs gear vs buzhash as drop-in candidates.
3. **Boundary hashing is not security.** None of these resists an adversary crafting inputs;
   MIN/MAX and the staircase bound the damage, and the per-level salt decorrelates levels — not
   attackers. Collision resistance lives one job over, in the address hash.

And the engine's measured verdict on the *cost* axis
([boundary study](../docs/write-ups/chunker-boundary-detection-study.md)): the whole chunker is
~1.6% of ingest, so none of the per-byte cost differences above moves the end-to-end needle —
**geometry, not compute, is the lever**. Choose by horizon and geometry; speed is a tiebreak.